# Stage 1 - Data Inspection

Inspect the raw World Cup datasets and verify the Stage 1 cleaning pipeline
(`src/ingest.py` + `src/clean.py`).

This notebook is **inspection only** - no feature engineering, modeling, or
simulation. Goalscorers are reviewed but not used as predictive features yet.

In [1]:
import sys
from pathlib import Path

# Make the project root importable so `import src...` works from notebooks/.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd

from src import config, ingest, clean

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

## 1. Load raw datasets

In [2]:
raw = ingest.load_all()
for name, df in raw.items():
    print(f"{name:<14} shape={df.shape}")

results        shape=(49472, 9)
goalscorers    shape=(47601, 8)
shootouts      shape=(678, 5)
eloratings     shape=(6678, 4)
former_names   shape=(36, 4)


### Columns & dtypes per raw dataset

In [3]:
for name, df in raw.items():
    print("=" * 70)
    print(name)
    print("=" * 70)
    print(df.dtypes)
    print()

results
date          datetime64[ns]
home_team             object
away_team             object
home_score           float64
away_score           float64
tournament            object
city                  object
country               object
neutral                 bool
dtype: object

goalscorers
date         datetime64[ns]
home_team            object
away_team            object
team                 object
scorer               object
minute              float64
own_goal               bool
penalty                bool
dtype: object

shootouts
date             datetime64[ns]
home_team                object
away_team                object
winner                   object
first_shooter            object
dtype: object

eloratings
date      datetime64[ns]
team              object
rating           float64
change             int64
dtype: object

former_names
current               object
former                object
start_date    datetime64[ns]
end_date      datetime64[ns]
dtype: object



## 2. Date parsing check (mixed-format Elo dates)

`eloratings.csv` mixes ISO (`YYYY-MM-DD`) and US (`M/D/YYYY`) date formats.
A naive `pd.to_datetime` parses only a handful of rows; `parse_mixed_dates`
handles both. Below we confirm there are no unparseable dates left.

In [4]:
naive = pd.to_datetime(
    pd.read_csv(config.RAW_FILES["eloratings"])["date"], errors="coerce"
)
fixed = raw["eloratings"]["date"]
print("naive parse  - NaT count:", int(naive.isna().sum()))
print("fixed parse  - NaT count:", int(fixed.isna().sum()))
print("fixed range :", fixed.min(), "->", fixed.max())

naive parse  - NaT count: 6634
fixed parse  - NaT count: 0
fixed range : 1872-11-30 00:00:00 -> 2025-12-13 00:00:00


## 3. Split results: played history vs. unplayed 2026 fixtures

In [5]:
played, fixtures = clean.split_results(raw["results"])
print("played  :", played.shape, "|", played["date"].min(), "->", played["date"].max())
print("fixtures:", fixtures.shape, "|", fixtures["date"].min(), "->", fixtures["date"].max())
print("\nUnplayed 2026 fixtures (prediction targets):")
fixtures.head(10)

played  : (49400, 9) | 1872-11-30 00:00:00 -> 2026-06-09 00:00:00
fixtures: (72, 7) | 2026-06-11 00:00:00 -> 2026-06-27 00:00:00

Unplayed 2026 fixtures (prediction targets):


,date,home_team,away_team,tournament,city,country,neutral
0,2026-06-11,Mexico,South Africa,FIFA World Cup,Mexico City,Mexico,False
1,2026-06-11,South Korea,Czech Republic,FIFA World Cup,Zapopan,Mexico,True
2,2026-06-12,Canada,Bosnia and Herzegovina,FIFA World Cup,Toronto,Canada,False
3,2026-06-12,United States,Paraguay,FIFA World Cup,Inglewood,United States,False
4,2026-06-13,Qatar,Switzerland,FIFA World Cup,Santa Clara,United States,True
5,2026-06-13,Brazil,Morocco,FIFA World Cup,East Rutherford,United States,True
6,2026-06-13,Haiti,Scotland,FIFA World Cup,Foxborough,United States,True
7,2026-06-13,Australia,Turkey,FIFA World Cup,Vancouver,Canada,True
8,2026-06-14,Sweden,Tunisia,FIFA World Cup,Guadalupe,Mexico,True
9,2026-06-14,Netherlands,Japan,FIFA World Cup,Arlington,United States,True


In [6]:
# Sanity: played matches must have no missing scores.
assert played[["home_score", "away_score"]].isna().sum().sum() == 0
print("OK - no missing scores in played history")

OK - no missing scores in played history


## 4. Clean Elo ratings (drop null, drop rating == 0, sort)

In [7]:
elo_raw = raw["eloratings"]
elo_clean = clean.clean_eloratings(elo_raw)
print("raw   :", elo_raw.shape, "| null ratings:", int(elo_raw["rating"].isna().sum()),
      "| zero ratings:", int((elo_raw["rating"] == 0).sum()))
print("clean :", elo_clean.shape, "| null ratings:", int(elo_clean["rating"].isna().sum()),
      "| min rating:", elo_clean["rating"].min())
elo_clean.head()

[elo] kept 6646/6678 rows after dropping invalid records
raw   : (6678, 4) | null ratings: 31 | zero ratings: 1
clean : (6646, 4) | null ratings: 0 | min rating: 389.0


,date,team,rating,change
0,1977-07-24,Afghanistan,905.0,-11
1,2011-07-03,Afghanistan,837.0,18
2,2014-05-27,Afghanistan,1095.0,-13
3,2015-10-13,Afghanistan,1076.0,-6
4,2016-03-29,Afghanistan,1151.0,24


## 5. Basic duplicate checks

In [8]:
for name, df in raw.items():
    print(f"{name:<14} duplicate rows: {int(df.duplicated().sum())}")

results        duplicate rows: 0
goalscorers    duplicate rows: 82
shootouts      duplicate rows: 0
eloratings     duplicate rows: 0
former_names   duplicate rows: 0


## 6. Summary table across all cleaned datasets

In [9]:
outputs = clean.run()
outputs["summary"]

[elo] kept 6646/6678 rows after dropping invalid records



=== Stage 1 dataset summary ===
           dataset  rows  columns  total_nulls  duplicate_rows   date_min   date_max
    results_played 49400        9            0               0 1872-11-30 2026-06-09
     fixtures_2026    72        7            0               0 2026-06-11 2026-06-27
  eloratings_clean  6646        4            0               0 1872-11-30 2025-12-13
 goalscorers_clean 47519        8          195               0 1916-07-02 2026-03-31
   shootouts_clean   678        5          422               0 1967-08-22 2026-06-06
former_names_clean    36        4            0               0        NaT        NaT

Interim files written to: /Users/nguyenthingochong/Projects/worldcup-2026-prediction/data/interim


,dataset,rows,columns,total_nulls,duplicate_rows,date_min,date_max
0,results_played,49400,9,0,0,1872-11-30,2026-06-09
1,fixtures_2026,72,7,0,0,2026-06-11,2026-06-27
2,eloratings_clean,6646,4,0,0,1872-11-30,2025-12-13
3,goalscorers_clean,47519,8,195,0,1916-07-02,2026-03-31
4,shootouts_clean,678,5,422,0,1967-08-22,2026-06-06
5,former_names_clean,36,4,0,0,NaT,NaT


## 7. Verify interim artifacts on disk

In [10]:
for key, path in config.INTERIM_FILES.items():
    print(f"{key:<20} {'exists' if path.exists() else 'MISSING':<8} {path.name}")

results_played       exists   results_played.parquet
fixtures_2026        exists   fixtures_2026.parquet
eloratings_clean     exists   eloratings_clean.parquet
goalscorers_clean    exists   goalscorers_clean.parquet
shootouts_clean      exists   shootouts_clean.parquet
former_names_clean   exists   former_names_clean.parquet
summary              exists   dataset_summary.csv
